# Seed 0 reproduction v1
Run all cells from the top in a Colab GPU runtime.
This copy uses the uploaded multi-seed training procedure with seed 0.
Outputs use `seed0_reproduction_v1`; original seed-0 reference files remain read-only.
All four models retain the 10-epoch schedule. An interrupted reproduction resumes only its own checkpoints.
Recorded Seed1 environment and runtime checks remain enabled.


# Seed2 training and clean-validation preparation

Run all cells in order in a Colab T4 GPU runtime. Change `SEED` only in the parameter cell. Four models run sequentially. No test images, test inference, or test-time threshold fitting are performed.

Exact Seed0 source snapshots and training settings are reused. RGB and geometry models retain their historical loader and checkpoint conventions. Each completed epoch is saved to Drive. After a runtime reset, run all cells again: completed epochs resume from saved checkpoints; existing prediction and threshold files are verified and reused.

This is a separate reusable notebook. The original Seed1 notebook remains the experiment record. Drive mounting may request authorization. Colab disconnections can interrupt execution; an unfinished epoch is rerun from its beginning.


In [ ]:
# Use seed 0 with an independent experiment identifier.
SEED = 0
RUN_TAG = f"seed{SEED}_reproduction_v1"
MODEL_NAMES = ("r18", "r50", "m3", "oracle")
PROJECT_PATH = "/content/drive/MyDrive/FacialVisualProfile"

## 1. Runtime and shared helpers

In [ ]:
from google.colab import drive
from pathlib import Path
from collections import Counter
import copy
import gc
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
import platform
import random
import shutil
import sqlite3
import sys
import time
import zipfile

import numpy as np
import torch
from torch import nn
from torch.torch_version import TorchVersion
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, average_precision_score

assert type(SEED) is int and SEED >= 0
assert MODEL_NAMES == ("r18", "r50", "m3", "oracle")
drive.mount("/content/drive")
PROJECT = Path(PROJECT_PATH)
assert PROJECT.is_dir(), f"Missing project: {PROJECT}"
assert torch.cuda.is_available(), "Select a GPU runtime before Run all"
DATA_ROOT = Path("/content/multiseed_shared_data/raw")
WORKSPACE_ROOT = Path("/content/multiseed_workspaces")
CACHE_PATH = DATA_ROOT.parent / "mtcnn_clean.sqlite"
ENVIRONMENT = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torchvision": importlib.metadata.version("torchvision"),
    "numpy": importlib.metadata.version("numpy"),
    "Pillow": importlib.metadata.version("Pillow"),
    "scikit-learn": importlib.metadata.version("scikit-learn"),
}
RUNTIME = {
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
}
# Keep the recorded Seed1 environment for the replication.
for name in MODEL_NAMES:
    cfg = json.loads((PROJECT / "runs" / f"{name}_seed1/config.json").read_text())
    assert ENVIRONMENT == cfg["environment"], f"Environment differs from {name} Seed1"
    assert RUNTIME == cfg["runtime"], f"CUDA, cuDNN or GPU differs from {name} Seed1"

def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, allow_nan=False) + "\n").encode("utf-8")

def write_json_atomic(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_bytes(json_bytes(value))
    temporary.replace(path)

def preserve_json(path, value):
    path = Path(path)
    if path.exists():
        assert json.loads(path.read_text()) == value, f"Existing record differs; preserved: {path}"
    else:
        write_json_atomic(path, value)

def copy_verified(source, destination, expected=None):
    source, destination = Path(source), Path(destination)
    actual = sha_file(source)
    if expected is not None:
        assert actual == expected, f"Source differs: {source}"
    if destination.exists():
        assert sha_file(destination) == actual, f"Existing file differs; preserved: {destination}"
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(destination.name + ".partial")
        shutil.copyfile(source, temporary)
        assert sha_file(temporary) == actual
        temporary.replace(destination)
    return actual

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

def set_seed(value):
    random.seed(value)
    np.random.seed(value)
    torch.manual_seed(value)
    torch.cuda.manual_seed_all(value)

print("Environment:", ENVIRONMENT)
print("Runtime:", RUNTIME)
print("Seed:", SEED)

## 2. Prepare shared train/validation images and the existing MTCNN cache once

In [ ]:
annotation_hashes = {}
for filename in ("list_attr_celeba.txt", "list_eval_partition.txt", "list_landmarks_align_celeba.txt"):
    annotation_hashes[filename] = copy_verified(PROJECT / "data" / filename, DATA_ROOT / filename)
rows = [line.split() for line in (DATA_ROOT / "list_eval_partition.txt").read_text(encoding="utf-8-sig").splitlines()]
assert all(len(row) == 2 and row[1] in ("0", "1", "2") for row in rows)
partitions = dict(rows)
assert len(rows) == len(partitions) == 202599
assert Counter(partitions.values()) == {"0": 162770, "1": 19867, "2": 19962}
TRAIN_NAMES = [name for name, split in rows if split == "0"]
VALID_NAMES = [name for name, split in rows if split == "1"]
REFERENCE_CONFIGS = {}
reference_hashes = {}
for name in MODEL_NAMES:
    path = PROJECT / "runs" / f"{name}_seed0/config.json"
    cfg = json.loads(path.read_text())
    assert cfg["seed"] == 0
    assert cfg["epochs"] == 10 and cfg["batch_size"] == 64
    assert cfg["learning_rate"] == 0.0001 and cfg["weight_decay"] == 0.0
    assert cfg["optimizer"] == "Adam" and cfg["precision"] == "FP32"
    assert cfg["num_workers"] == 2
    assert cfg["selection_metric"] == "validation_mAP"
    assert cfg["epoch_seed_rule"] == "seed + epoch; epochs start at 1"
    assert cfg["cudnn_deterministic"] is True and cfg["cudnn_benchmark"] is False
    REFERENCE_CONFIGS[name] = cfg
    reference_hashes[name] = sha_file(path)
cache_hash = copy_verified(PROJECT / "cache/mtcnn_clean.sqlite", CACHE_PATH)
with sqlite3.connect(CACHE_PATH.resolve().as_uri() + "?mode=ro", uri=True) as connection:
    assert connection.execute("PRAGMA quick_check").fetchone()[0] == "ok"
    cache_config = json.loads(connection.execute("SELECT value FROM metadata WHERE key='config'").fetchone()[0])
    assert cache_config["partition_sha256"] == annotation_hashes["list_eval_partition.txt"]
    records = {name: (split, encoded) for name, split, encoded in connection.execute("SELECT name, split, result FROM detections")}
assert set(records) == set(partitions)
expected_zip_hash = "46fb89443c578308acf364d7d379fe1b9efb793042c0af734b6112e4fd3a8c74"
candidates = [DATA_ROOT.parent / "img_align_celeba.zip", Path("/content/seed1_shared_data/img_align_celeba.zip"), Path("/content/r50_validation_data/img_align_celeba.zip")]
archive_path = next((path for path in candidates if path.is_file()), candidates[0])
if not archive_path.exists():
    print("Copying original image archive...", flush=True)
    copy_verified(PROJECT / "data/img_align_celeba.zip", archive_path, expected_zip_hash)
assert sha_file(archive_path) == expected_zip_hash
image_root = DATA_ROOT / "img_align_celeba"
image_root.mkdir(parents=True, exist_ok=True)
counts = Counter()
with zipfile.ZipFile(archive_path) as archive:
    members = {}
    for member in archive.infolist():
        if member.is_dir():
            continue
        assert member.filename.lower().endswith(".jpg")
        name = Path(member.filename).name
        assert name not in members
        members[name] = member
    assert set(members) == set(partitions)
    selected_rows = [(name, split) for name, split in rows if split in ("0", "1")]
    for index, (name, split) in enumerate(selected_rows, 1):
        cached_split, encoded = records[name]
        assert cached_split == split
        result = json.loads(encoded)
        counts[(split, result["status"])] += 1
        content = archive.read(members[name])
        assert hashlib.sha256(content).hexdigest() == result["image_sha256"], f"Image differs: {name}"
        destination = image_root / name
        if destination.exists():
            assert destination.read_bytes() == content, f"Existing image differs: {name}"
        else:
            temporary = destination.with_name(name + ".partial")
            temporary.write_bytes(content)
            temporary.replace(destination)
        if index % 20000 == 0 or index == len(selected_rows):
            print(f"Verified train/validation images: {index}/{len(selected_rows)}", flush=True)
assert counts == Counter({("0", "ok"): 162521, ("0", "no_face"): 249, ("1", "ok"): 19835, ("1", "no_face"): 32})
SHARED_PREPARATION = {
    "image_zip_sha256": expected_zip_hash,
    "annotation_sha256": annotation_hashes,
    "reference_config_sha256": reference_hashes,
    "cache_sha256": cache_hash,
    "train_samples": 162770,
    "validation_samples": 19867,
    "test_evaluated": False,
}
SHARED_HASH = hashlib.sha256(json_bytes(SHARED_PREPARATION)).hexdigest()
del records, content
print("Shared images, annotation hashes and cache: PASS")
print("No test images extracted or detector inference performed")

## 3. Model setup and checkpoint restoration

In [ ]:
def prepare_run(model_name):
    reference = REFERENCE_CONFIGS[model_name]
    run_name = f"{model_name}_{RUN_TAG}"
    run_root = PROJECT / "runs" / run_name
    reference_root = PROJECT / "runs" / f"{model_name}_seed0"
    workspace = WORKSPACE_ROOT / run_name
    preparation = {"run_name": run_name, "seed": SEED, "shared": SHARED_PREPARATION}
    config = copy.deepcopy(reference)
    config.pop("git_commit", None)
    config.update({
        "run_name": run_name, "seed": SEED,
        "environment": ENVIRONMENT, "runtime": RUNTIME,
        "reference_run": f"{model_name}_seed0",
        "reference_config_sha256": reference_hashes[model_name],
        "reference_environment": reference["environment"],
        "selection_tie_rule": "earliest epoch",
        "data_preparation_sha256": hashlib.sha256(json_bytes(preparation)).hexdigest(),
    })
    if "git_commit" in reference:
        config["reference_source_commit"] = reference["git_commit"]
    if model_name == "m3":
        config["cache_sha256"] = SHARED_PREPARATION["cache_sha256"]
    # Validate every destination before creating new files.
    writes = {}
    for relative, expected in config["source_sha256"].items():
        rel = Path(relative)
        assert not rel.is_absolute() and ".." not in rel.parts
        source = reference_root / "source" / rel
        assert sha_file(source) == expected
        data = source.read_bytes()
        for destination in (workspace / rel, run_root / "source" / rel):
            if destination.exists():
                assert destination.read_bytes() == data, f"Existing source differs: {destination}"
            writes[destination] = data
    for filename, value in (("config.json", config), ("data_preparation.json", preparation)):
        path = run_root / filename
        if path.exists():
            assert json.loads(path.read_text()) == value, f"Existing run differs; preserved: {path}"
    for path, data in writes.items():
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(data)
    preserve_json(run_root / "config.json", config)
    preserve_json(run_root / "data_preparation.json", preparation)
    for package in ("datasets", "models", "evaluation"):
        initializer = workspace / package / "__init__.py"
        if not initializer.exists():
            initializer.write_text("")
    for name in list(sys.modules):
        if any(name == package or name.startswith(package + ".") for package in ("datasets", "models", "evaluation")):
            del sys.modules[name]
    os.chdir(workspace)
    if str(workspace) in sys.path:
        sys.path.remove(str(workspace))
    sys.path.insert(0, str(workspace))
    importlib.invalidate_caches()
    dataset_module = importlib.import_module("datasets.celeba_predicted" if model_name == "m3" else "datasets.celeba")
    model_module = importlib.import_module({"r18": "models.resnet18", "r50": "models.resnet50", "m3": "models.resnet18_geometry", "oracle": "models.resnet18_geometry"}[model_name])
    metric_module = importlib.import_module("evaluation.attributes")
    for relative, expected in config["source_sha256"].items():
        if relative.endswith(".py"):
            name = relative[:-3].replace("/", ".")
            if name in sys.modules:
                assert sha_file(sys.modules[name].__file__) == expected
    if model_name == "m3":
        train_dataset = dataset_module.CelebAPredictedAttributes(CACHE_PATH, split="train", root=DATA_ROOT, failure_policy="mask")
        valid_dataset = dataset_module.CelebAPredictedAttributes(CACHE_PATH, split="valid", root=DATA_ROOT, failure_policy="mask")
    else:
        train_dataset = dataset_module.CelebAAttributes(split="train", root=DATA_ROOT)
        valid_dataset = dataset_module.CelebAAttributes(split="valid", root=DATA_ROOT)
    assert train_dataset.names == TRAIN_NAMES and valid_dataset.names == VALID_NAMES
    assert train_dataset.attribute_names == valid_dataset.attribute_names == config["attribute_names"]
    set_seed(SEED)
    torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
    torch.backends.cudnn.benchmark = config["cudnn_benchmark"]
    checkpoints = sorted(run_root.glob("epoch_[0-9][0-9].pt"))
    cls = getattr(model_module, {"r18": "ResNet18Attributes", "r50": "ResNet50Attributes", "m3": "ResNet18GeometryAttributes", "oracle": "ResNet18GeometryAttributes"}[model_name])
    device = torch.device("cuda")
    model = cls(num_attributes=24, pretrained=not bool(checkpoints)).to(device)
    assert all(p.requires_grad for p in model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    rgb = model_name in ("r18", "r50")
    if checkpoints:
        with torch.serialization.safe_globals([TorchVersion]):
            state = torch.load(checkpoints[-1], map_location="cpu", weights_only=True)
        assert state["config"] == config
        assert [row["epoch"] for row in state["history"]] == list(range(1, state["epoch"] + 1))
        assert checkpoints[-1].name == f"epoch_{state['epoch']:02d}.pt"
        model.load_state_dict(state["model_state" if rgb else "model_state_dict"], strict=True)
        optimizer.load_state_dict(state["optimizer_state" if rgb else "optimizer_state_dict"])
        del state
        print("Restored:", checkpoints[-1].name)
    else:
        state = {"epoch": 0, "config": config, "history": [],
                 "model_state" if rgb else "model_state_dict": model.state_dict(),
                 "optimizer_state" if rgb else "optimizer_state_dict": optimizer.state_dict()}
        if not rgb:
            state["best"] = None if model_name == "m3" else {"epoch": None, "map": None}
        path = run_root / "epoch_00.pt"
        temporary = path.with_name(path.name + ".partial")
        torch.save(state, temporary)
        temporary.replace(path)
        del state
        print("Fresh ImageNet initialization saved:", path)
    model.eval()
    samples = [valid_dataset[i] for i in range(4)]
    images = torch.stack([s[0] for s in samples]).to(device)
    with torch.inference_mode():
        if rgb:
            logits = model(images)
        else:
            points = torch.stack([s[2] for s in samples]).to(device)
            valid = torch.tensor([s[4] if model_name == "m3" else True for s in samples], dtype=torch.bool, device=device)
            logits = model(images, points, valid)
    assert logits.shape == (4, 24) and torch.isfinite(logits).all()
    print(run_name, "setup and forward: PASS")
    return {"model_name": model_name, "model": model, "optimizer": optimizer,
            "criterion": nn.BCEWithLogitsLoss(), "config": config,
            "train_dataset": train_dataset, "valid_dataset": valid_dataset,
            "run_root": run_root, "device": device,
            "metric_function": metric_module.evaluate_attributes}

## 4. Historical training loops (RGB, predicted geometry, GT geometry)

In [ ]:
def train_rgb(context, project):
    model = context["model"]
    optimizer = context["optimizer"]
    criterion = context["criterion"]
    train_dataset = context["train_dataset"]
    valid_dataset = context["valid_dataset"]
    config = context["config"]
    device = context["device"]
    metric_function = context["metric_function"]

    run_name = config["run_name"]
    model_name = run_name.split("_seed")[0]
    assert model_name in ("r18", "r50")
    assert run_name == f"{model_name}_{RUN_TAG}"
    assert torch.cuda.is_available()

    run_root = Path(project) / "runs" / run_name
    assert json.loads(
        (run_root / "config.json").read_text(encoding="utf-8")
    ) == config, "Run configuration mismatch"

    for relative, expected_hash in config["source_sha256"].items():
        assert sha_file(run_root / "source" / relative) == expected_hash, (
            f"Archived source differs: {relative}"
        )

    assert config["optimizer"] == "Adam"
    assert config["precision"] == "FP32"
    assert config["f1_threshold"] == 0.5
    assert config["selection_metric"] == "validation_mAP"
    assert config["epoch_seed_rule"] == (
        "seed + epoch; epochs start at 1"
    )

    torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
    torch.backends.cudnn.benchmark = config["cudnn_benchmark"]

    def make_loader(dataset, shuffle, epoch_seed):
        options = {
            "batch_size": config["batch_size"],
            "shuffle": shuffle,
            "num_workers": config["num_workers"],
            "worker_init_fn": seed_worker,
            "pin_memory": True,
            "drop_last": False,
        }
        # Preserve each model's historical DataLoader configuration.
        if shuffle or model_name == "r18":
            options["generator"] = (
                torch.Generator().manual_seed(epoch_seed)
            )
        return DataLoader(dataset, **options)

    def save_summaries(history):
        write_json_atomic(run_root / "history.json", history)
        if not history:
            return None

        # Equal mAP values retain the earliest epoch.
        best = max(history, key=lambda row: row["validation"]["map"])
        summary = {
            "epoch": best["epoch"],
            "checkpoint": f"epoch_{best['epoch']:02d}.pt",
            "validation_map": best["validation"]["map"],
            "validation_macro_f1": best["validation"]["macro_f1"],
        }
        write_json_atomic(run_root / "best.json", summary)
        return summary

    # Reload the latest completed epoch on every invocation.
    checkpoints = sorted(run_root.glob("epoch_[0-9][0-9].pt"))
    assert checkpoints, "Saved checkpoint is missing"
    latest = checkpoints[-1]

    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            latest, map_location="cpu", weights_only=True
        )

    assert checkpoint["config"] == config
    model.load_state_dict(checkpoint["model_state"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state"])

    history = checkpoint["history"]
    completed_epoch = checkpoint["epoch"]
    assert [row["epoch"] for row in history] == list(
        range(1, completed_epoch + 1)
    ), "Checkpoint history is not continuous"
    assert 0 <= completed_epoch <= config["epochs"]
    del checkpoint

    config["environment"]["torch"] = str(
        config["environment"]["torch"]
    )
    save_summaries(history)

    print("Run:", run_name, flush=True)
    print("Restored:", latest.name, flush=True)
    print("Next epoch:", completed_epoch + 1, flush=True)

    for epoch in range(completed_epoch + 1, config["epochs"] + 1):
        epoch_seed = config["seed"] + epoch
        random.seed(epoch_seed)
        np.random.seed(epoch_seed)
        torch.manual_seed(epoch_seed)
        torch.cuda.manual_seed_all(epoch_seed)

        train_loader = make_loader(train_dataset, True, epoch_seed)
        valid_loader = make_loader(valid_dataset, False, epoch_seed)

        torch.cuda.synchronize()
        started = time.perf_counter()
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        progress = tqdm(
            train_loader,
            desc=f"{run_name} epoch {epoch:02d} train",
        )
        for images, targets, _, _ in progress:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, targets)
            assert torch.isfinite(loss), "Non-finite training loss"

            loss.backward()
            optimizer.step()

            count = images.size(0)
            train_loss_sum += loss.item() * count
            train_count += count

            if train_count % (config["batch_size"] * 50) == 0:
                progress.set_postfix(
                    BCE=f"{train_loss_sum / train_count:.6f}"
                )

        assert train_count == len(train_dataset)

        model.eval()
        valid_loss_sum = 0.0
        valid_count = 0
        logits_parts = []
        target_parts = []

        with torch.inference_mode():
            for images, targets, _, _ in tqdm(
                valid_loader,
                desc=f"{run_name} epoch {epoch:02d} valid",
            ):
                images = images.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                logits = model(images)
                loss = criterion(logits, targets)

                assert torch.isfinite(logits).all()
                assert torch.isfinite(loss), "Non-finite validation loss"

                count = images.size(0)
                valid_loss_sum += loss.item() * count
                valid_count += count
                logits_parts.append(logits.cpu())
                target_parts.append(targets.cpu())

        assert valid_count == len(valid_dataset)

        metrics = metric_function(
            torch.cat(logits_parts).numpy(),
            torch.cat(target_parts).numpy(),
            threshold=config["f1_threshold"],
        )
        del logits_parts, target_parts

        torch.cuda.synchronize()
        record = {
            "epoch": epoch,
            "train_loss": train_loss_sum / train_count,
            "valid_loss": valid_loss_sum / valid_count,
            "validation": metrics,
            "seconds_excluding_save": time.perf_counter() - started,
        }
        history.append(record)

        checkpoint_path = run_root / f"epoch_{epoch:02d}.pt"
        assert not checkpoint_path.exists(), (
            f"Completed checkpoint already exists: {checkpoint_path}"
        )
        temporary = checkpoint_path.with_name(
            checkpoint_path.name + ".partial"
        )
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "config": config,
            "history": history,
        }, temporary)
        temporary.replace(checkpoint_path)
        best = save_summaries(history)

        print(
            f"Epoch {epoch:02d} saved | "
            f"train BCE={record['train_loss']:.6f} | "
            f"valid BCE={record['valid_loss']:.6f} | "
            f"F1@0.5={metrics['macro_f1']:.6f} | "
            f"mAP={metrics['map']:.6f} | "
            f"{record['seconds_excluding_save'] / 60:.1f} min | "
            f"best epoch={best['epoch']}",
            flush=True,
        )

    best = save_summaries(history)
    print("Selected checkpoint:", run_root / best["checkpoint"])
    print("RGB training: COMPLETE")
    print("Validation prediction export and threshold fitting remain pending")
    print("No test evaluation performed")
    return history, best

def train_m3(
    model, optimizer, criterion, config,
    train_dataset, valid_dataset, run_root, device,
):
    saved_config = json.loads(
        (run_root / "config.json").read_text(encoding="utf-8")
    )
    assert saved_config == config, "Run configuration mismatch"
    assert config["run_name"] == f"m3_{RUN_TAG}"
    assert config["optimizer"] == "Adam"
    assert config["precision"] == "FP32"
    assert config["f1_threshold"] == 0.5
    assert config["selection_metric"] == "validation_mAP"
    assert config["selection_tie_rule"] == "earliest epoch"
    assert config["epoch_seed_rule"] == (
        "seed + epoch; epochs start at 1"
    )
    assert len(train_dataset) == 162770
    assert len(valid_dataset) == 19867
    assert train_dataset.failure_policy == "mask"
    assert valid_dataset.failure_policy == "mask"

    for relative, expected in config["source_sha256"].items():
        assert sha_file(run_root / "source" / relative) == expected, (
            f"Archived source differs: {relative}"
        )

    torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
    torch.backends.cudnn.benchmark = config["cudnn_benchmark"]

    def read_metric(metrics, name):
        matches = [
            value for key, value in metrics.items()
            if key.replace("_", "").lower() == name
        ]
        assert len(matches) == 1, (
            f"Unexpected metric keys: {list(metrics)}"
        )
        value = float(matches[0])
        assert math.isfinite(value), f"Non-finite metric: {name}"
        return value

    def select_best(history):
        if not history:
            return None
        # Python max preserves the earliest record on exact ties.
        row = max(history, key=lambda item: item["validation_mAP"])
        return {
            "epoch": row["epoch"],
            "checkpoint": f"epoch_{row['epoch']:02d}.pt",
            "validation_bce": row["validation_bce"],
            "validation_macro_f1": row["validation_macro_f1"],
            "validation_mAP": row["validation_mAP"],
            "f1_threshold": config["f1_threshold"],
        }

    probe_metrics = evaluate_attributes(
        np.array([[-10.0] * 24, [10.0] * 24]),
        np.array([[0] * 24, [1] * 24]),
        threshold=0.5,
    )
    assert read_metric(probe_metrics, "macrof1") == 1.0
    assert read_metric(probe_metrics, "map") == 1.0

    # Reload saved state on every invocation.
    checkpoints = sorted(run_root.glob("epoch_[0-9][0-9].pt"))
    assert checkpoints, "Initialization checkpoint is missing"
    latest = checkpoints[-1]

    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            latest, map_location="cpu", weights_only=True
        )

    assert checkpoint["config"] == config
    completed_epoch = int(checkpoint["epoch"])
    assert latest.name == f"epoch_{completed_epoch:02d}.pt"
    assert 0 <= completed_epoch <= config["epochs"]

    history = checkpoint["history"]
    assert [row["epoch"] for row in history] == list(
        range(1, completed_epoch + 1)
    ), "Checkpoint history is not continuous"

    best = select_best(history)
    assert checkpoint["best"] == best, "Checkpoint selection mismatch"

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    del checkpoint
    torch.cuda.empty_cache()

    # Checkpoint history is authoritative after interrupted summary writes.
    write_json_atomic(run_root / "history.json", history)
    if best is not None:
        write_json_atomic(run_root / "best.json", best)

    # Preserve the historical M3 loader settings: no explicit generator
    # or custom worker initializer.
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=config["num_workers"],
        pin_memory=True,
        drop_last=False,
    )

    print("Run:", config["run_name"], flush=True)
    print("Restored:", latest.name, flush=True)
    print("Completed epochs:", completed_epoch, flush=True)
    print("Next epoch:", completed_epoch + 1, flush=True)

    for epoch in range(completed_epoch + 1, config["epochs"] + 1):
        epoch_seed = config["seed"] + epoch
        random.seed(epoch_seed)
        np.random.seed(epoch_seed)
        torch.manual_seed(epoch_seed)
        torch.cuda.manual_seed_all(epoch_seed)

        train_loader = DataLoader(
            train_dataset,
            batch_size=config["batch_size"],
            shuffle=True,
            num_workers=config["num_workers"],
            pin_memory=True,
            drop_last=False,
        )

        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        started = time.perf_counter()

        model.train()
        train_loss_sum = 0.0
        train_count = 0
        train_missing = 0

        progress = tqdm(
            train_loader,
            desc=f"{config['run_name']} epoch {epoch:02d} train",
        )
        for images, labels, points, names, valid in progress:
            count = images.shape[0]
            train_missing += int((~valid).sum().item())

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            points = points.to(device, non_blocking=True)
            valid = valid.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images, points, valid)
            loss = criterion(logits, labels)
            assert torch.isfinite(loss), (
                f"Non-finite training loss in epoch {epoch}"
            )

            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * count
            train_count += count
            progress.set_postfix(
                BCE=f"{train_loss_sum / train_count:.4f}"
            )

        assert train_count == 162770
        assert train_missing == 249

        model.eval()
        valid_loss_sum = 0.0
        valid_count = 0
        valid_missing = 0
        logits_parts = []
        target_parts = []

        with torch.inference_mode():
            for images, labels, points, names, valid in tqdm(
                valid_loader,
                desc=f"{config['run_name']} epoch {epoch:02d} valid",
            ):
                count = images.shape[0]
                valid_missing += int((~valid).sum().item())
                target_parts.append(labels.numpy())

                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                points = points.to(device, non_blocking=True)
                valid = valid.to(device, non_blocking=True)

                logits = model(images, points, valid)
                loss = criterion(logits, labels)
                assert torch.isfinite(logits).all()
                assert torch.isfinite(loss)

                valid_loss_sum += loss.item() * count
                valid_count += count
                logits_parts.append(logits.cpu().numpy())

        assert valid_count == 19867
        assert valid_missing == 32

        metrics = evaluate_attributes(
            np.concatenate(logits_parts),
            np.concatenate(target_parts),
            threshold=config["f1_threshold"],
        )
        macro_f1 = read_metric(metrics, "macrof1")
        validation_map = read_metric(metrics, "map")

        torch.cuda.synchronize()
        record = {
            "epoch": epoch,
            "train_bce": train_loss_sum / train_count,
            "validation_bce": valid_loss_sum / valid_count,
            "validation_macro_f1": macro_f1,
            "validation_mAP": validation_map,
            "train_images": train_count,
            "validation_images": valid_count,
            "train_missing_geometry": train_missing,
            "validation_missing_geometry": valid_missing,
            "elapsed_seconds": time.perf_counter() - started,
            "peak_allocated_gpu_gib": (
                torch.cuda.max_memory_allocated() / 1024**3
            ),
        }
        history.append(record)
        best = select_best(history)

        checkpoint_path = run_root / f"epoch_{epoch:02d}.pt"
        assert not checkpoint_path.exists(), (
            f"Completed checkpoint already exists: {checkpoint_path}"
        )
        temporary = checkpoint_path.with_name(
            checkpoint_path.name + ".partial"
        )
        torch.save({
            "epoch": epoch,
            "config": config,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "history": history,
            "best": best,
        }, temporary)
        temporary.replace(checkpoint_path)

        write_json_atomic(run_root / "history.json", history)
        write_json_atomic(run_root / "best.json", best)

        print(
            f"Epoch {epoch:02d} saved | "
            f"train BCE={record['train_bce']:.6f} | "
            f"valid BCE={record['validation_bce']:.6f} | "
            f"F1@0.5={macro_f1:.6f} | "
            f"mAP={validation_map:.6f} | "
            f"{record['elapsed_seconds'] / 60:.1f} min | "
            f"best epoch={best['epoch']}",
            flush=True,
        )
        del train_loader, logits_parts, target_parts

    print("Selected checkpoint:", run_root / best["checkpoint"])
    print("M3 training: COMPLETE")
    print("Clean-validation prediction export and threshold fitting remain pending")
    print("No test evaluation performed")
    return history, best

def train_oracle(
    model, optimizer, criterion, config,
    train_dataset, valid_dataset, run_root, device,
):
    assert config["run_name"] == f"oracle_{RUN_TAG}"
    assert json.loads(
        (run_root / "config.json").read_text(encoding="utf-8")
    ) == config
    assert config["optimizer"] == "Adam"
    assert config["precision"] == "FP32"
    assert config["f1_threshold"] == 0.5
    assert config["selection_metric"] == "validation_mAP"
    assert config["selection_tie_rule"] == "earliest epoch"
    assert config["epoch_seed_rule"] == (
        "seed + epoch; epochs start at 1"
    )
    assert config["geometry_source"] == (
        "CelebA aligned GT; shared preprocessing"
    )
    assert type(train_dataset).__name__ == "CelebAAttributes"
    assert type(valid_dataset).__name__ == "CelebAAttributes"
    assert len(train_dataset) == 162770
    assert len(valid_dataset) == 19867
    assert train_dataset.attribute_names == config["attribute_names"]
    assert valid_dataset.attribute_names == config["attribute_names"]

    for relative, expected in config["source_sha256"].items():
        assert sha_file(run_root / "source" / relative) == expected, (
            f"Archived source differs: {relative}"
        )

    torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
    torch.backends.cudnn.benchmark = config["cudnn_benchmark"]

    def select_best(history):
        if not history:
            return {"epoch": None, "map": None}
        row = max(history, key=lambda item: item["validation_map"])
        return {"epoch": row["epoch"], "map": row["validation_map"]}

    def make_loader(dataset, shuffle):
        # Preserve the historical Oracle loader settings.
        return DataLoader(
            dataset,
            batch_size=config["batch_size"],
            shuffle=shuffle,
            num_workers=config["num_workers"],
            pin_memory=True,
            drop_last=False,
            worker_init_fn=seed_worker,
        )

    checkpoints = sorted(run_root.glob("epoch_[0-9][0-9].pt"))
    assert checkpoints, "Saved checkpoint is missing"
    latest = checkpoints[-1]

    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            latest, map_location="cpu", weights_only=True
        )
    assert checkpoint["config"] == config

    completed_epoch = checkpoint["epoch"]
    history = checkpoint["history"]
    assert latest.name == f"epoch_{completed_epoch:02d}.pt"
    assert 0 <= completed_epoch <= config["epochs"]
    assert [row["epoch"] for row in history] == list(
        range(1, completed_epoch + 1)
    )

    best = select_best(history)
    assert checkpoint["best"] == best, "Checkpoint selection mismatch"

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    del checkpoint
    torch.cuda.empty_cache()

    write_json_atomic(run_root / "history.json", history)
    if history:
        write_json_atomic(run_root / "best.json", best)

    print("Run:", config["run_name"], flush=True)
    print("Restored:", latest.name, flush=True)
    print("Completed epochs:", completed_epoch, flush=True)
    print("Next epoch:", completed_epoch + 1, flush=True)
    print("Geometry: GT; all samples valid", flush=True)

    for epoch in range(completed_epoch + 1, config["epochs"] + 1):
        epoch_seed = config["seed"] + epoch
        random.seed(epoch_seed)
        np.random.seed(epoch_seed)
        torch.manual_seed(epoch_seed)
        torch.cuda.manual_seed_all(epoch_seed)

        torch.cuda.synchronize()
        started = time.perf_counter()
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for images, targets, points, names in tqdm(
            make_loader(train_dataset, True),
            desc=f"{config['run_name']} epoch {epoch:02d} train",
        ):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            points = points.to(device, non_blocking=True)
            valid = torch.ones(
                images.shape[0], dtype=torch.bool, device=device
            )

            optimizer.zero_grad(set_to_none=True)
            logits = model(images, points, valid)
            loss = criterion(logits, targets)
            assert torch.isfinite(loss), "Non-finite training loss"

            loss.backward()
            optimizer.step()

            count = images.shape[0]
            train_loss_sum += loss.item() * count
            train_count += count

        assert train_count == 162770

        model.eval()
        valid_loss_sum = 0.0
        valid_count = 0
        logits_parts = []
        target_parts = []
        validation_names = []

        with torch.inference_mode():
            for images, targets, points, names in tqdm(
                make_loader(valid_dataset, False),
                desc=f"{config['run_name']} epoch {epoch:02d} valid",
            ):
                images = images.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                points = points.to(device, non_blocking=True)
                valid = torch.ones(
                    images.shape[0], dtype=torch.bool, device=device
                )

                logits = model(images, points, valid)
                loss = criterion(logits, targets)
                assert torch.isfinite(logits).all()
                assert torch.isfinite(loss)

                count = images.shape[0]
                valid_loss_sum += loss.item() * count
                valid_count += count
                logits_parts.append(logits.cpu().numpy())
                target_parts.append(targets.cpu().numpy())
                validation_names.extend(names)

        assert valid_count == 19867
        assert validation_names == valid_dataset.names

        metrics = evaluate_attributes(
            np.concatenate(logits_parts),
            np.concatenate(target_parts),
            threshold=config["f1_threshold"],
        )
        assert np.isfinite(metrics["macro_f1"])
        assert np.isfinite(metrics["map"])

        torch.cuda.synchronize()
        record = {
            "epoch": epoch,
            "train_bce": train_loss_sum / train_count,
            "validation_bce": valid_loss_sum / valid_count,
            "validation_macro_f1_at_0.5": float(metrics["macro_f1"]),
            "validation_map": float(metrics["map"]),
            "seconds": time.perf_counter() - started,
        }
        history.append(record)
        best = select_best(history)

        checkpoint_path = run_root / f"epoch_{epoch:02d}.pt"
        assert not checkpoint_path.exists(), (
            f"Completed checkpoint already exists: {checkpoint_path}"
        )
        temporary = checkpoint_path.with_name(
            checkpoint_path.name + ".partial"
        )
        torch.save({
            "epoch": epoch,
            "config": config,
            "history": history,
            "best": best,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        }, temporary)
        temporary.replace(checkpoint_path)

        write_json_atomic(run_root / "history.json", history)
        write_json_atomic(run_root / "best.json", best)

        print(
            f"Epoch {epoch:02d} saved | "
            f"train BCE={record['train_bce']:.6f} | "
            f"valid BCE={record['validation_bce']:.6f} | "
            f"F1@0.5={metrics['macro_f1']:.6f} | "
            f"mAP={metrics['map']:.6f} | "
            f"{record['seconds'] / 60:.1f} min | "
            f"best epoch={best['epoch']}",
            flush=True,
        )
        del logits_parts, target_parts

    assert len(history) == config["epochs"]
    print(
        "Selected checkpoint:",
        run_root / f"epoch_{best['epoch']:02d}.pt",
    )
    print("Oracle training: COMPLETE")
    print("Validation prediction export and threshold fitting remain pending")
    print("No test evaluation performed")
    return history, best

## 5. Selected-checkpoint validation export and clean-validation thresholds

In [ ]:
def selected_record(context):
    root = context["run_root"]
    config = context["config"]
    history = json.loads((root / "history.json").read_text())
    assert [row["epoch"] for row in history] == list(range(1, config["epochs"] + 1))
    kind = context["model_name"]
    metric = (lambda row: row["validation"]["map"]) if kind in ("r18", "r50") else (lambda row: row["validation_mAP"] if kind == "m3" else row["validation_map"])
    row = max(history, key=metric)
    expected_map = metric(row)
    expected_f1 = row["validation"]["macro_f1"] if kind in ("r18", "r50") else row["validation_macro_f1"] if kind == "m3" else row["validation_macro_f1_at_0.5"]
    best = json.loads((root / "best.json").read_text())
    assert best["epoch"] == row["epoch"]
    for epoch in range(config["epochs"] + 1):
        assert (root / f"epoch_{epoch:02d}.pt").is_file()
    return history, row["epoch"], float(expected_f1), float(expected_map)

def export_and_freeze(context):
    kind = context["model_name"]
    root, config = context["run_root"], context["config"]
    model, device = context["model"], context["device"]
    dataset = context["valid_dataset"]
    metric_function = context["metric_function"]
    history, epoch, expected_f1, expected_map = selected_record(context)
    checkpoint_path = root / f"epoch_{epoch:02d}.pt"
    prediction_path = root / "validation_predictions.npz"
    audit_path = root / "validation_export_audit.json"
    hashes = {"checkpoint_sha256": sha_file(checkpoint_path),
              "config_sha256": sha_file(root / "config.json")}
    with torch.serialization.safe_globals([TorchVersion]):
        state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    assert state["config"] == config and state["epoch"] == epoch
    assert state["history"] == history[:epoch]
    model.load_state_dict(state["model_state" if kind in ("r18", "r50") else "model_state_dict"], strict=True)
    del state
    model.eval()
    if prediction_path.exists() and audit_path.exists():
        saved_audit = json.loads(audit_path.read_text())
        for key, expected in hashes.items():
            assert saved_audit[key] == expected
        assert saved_audit["prediction_sha256"] == sha_file(prediction_path)
        with np.load(prediction_path, allow_pickle=False) as saved:
            payload = {key: saved[key].copy() for key in saved.files}
        print(config["run_name"], "saved validation predictions: REUSED")
    else:
        options = {"batch_size": config["batch_size"], "shuffle": False,
                   "num_workers": config["num_workers"], "pin_memory": True, "drop_last": False}
        if kind != "m3":
            options["worker_init_fn"] = seed_worker
        if kind == "r18":
            options["generator"] = torch.Generator().manual_seed(config["seed"] + epoch)
        logits_parts, target_parts, valid_parts, names = [], [], [], []
        with torch.inference_mode():
            for batch in tqdm(DataLoader(dataset, **options), desc=f"{config['run_name']} selected validation"):
                images, targets, points, filenames = batch[:4]
                images = images.to(device, non_blocking=True)
                valid = batch[4] if kind == "m3" else torch.ones(len(filenames), dtype=torch.bool)
                if kind in ("r18", "r50"):
                    logits = model(images)
                else:
                    logits = model(images, points.to(device, non_blocking=True), valid.to(device, non_blocking=True))
                assert logits.shape == (len(filenames), 24) and torch.isfinite(logits).all()
                logits_parts.append(logits.cpu().numpy())
                target_parts.append(targets.numpy())
                valid_parts.append(valid.numpy())
                names.extend(filenames)
        payload = {"filenames": np.asarray(names), "logits": np.concatenate(logits_parts),
                   "targets": np.concatenate(target_parts), "attribute_names": np.asarray(config["attribute_names"])}
        if kind in ("m3", "oracle"):
            payload["geometry_valid"] = np.concatenate(valid_parts)
        del logits_parts, target_parts, valid_parts
    assert payload["filenames"].tolist() == VALID_NAMES == dataset.names
    assert payload["logits"].shape == payload["targets"].shape == (19867, 24)
    assert np.isfinite(payload["logits"]).all()
    assert np.isin(payload["targets"], [0, 1]).all()
    assert payload["attribute_names"].tolist() == config["attribute_names"]
    masked = 0
    if kind in ("m3", "oracle"):
        assert payload["geometry_valid"].dtype == np.bool_
        assert payload["geometry_valid"].shape == (19867,)
        masked = int((~payload["geometry_valid"]).sum())
        assert masked == (32 if kind == "m3" else 0)
    with np.load(PROJECT / "runs/r18_seed0/day7/validation_predictions.npz", allow_pickle=False) as reference:
        for key in ("filenames", "targets", "attribute_names"):
            assert np.array_equal(payload[key], reference[key]), f"Historical validation alignment differs: {key}"
    metrics = metric_function(payload["logits"], payload["targets"], threshold=0.5)
    assert abs(metrics["macro_f1"] - expected_f1) <= 1e-10
    assert abs(metrics["map"] - expected_map) <= 1e-10
    if prediction_path.exists():
        with np.load(prediction_path, allow_pickle=False) as existing:
            assert set(existing.files) == set(payload)
            for key, value in payload.items():
                assert np.array_equal(existing[key], value), f"Existing predictions differ; preserved: {key}"
    else:
        temporary = prediction_path.with_name(prediction_path.name + ".partial")
        with temporary.open("wb") as stream:
            np.savez_compressed(stream, **payload)
        temporary.replace(prediction_path)
    audit = {"run_name": config["run_name"], "checkpoint": checkpoint_path.name,
             "selected_epoch": epoch, **hashes,
             "prediction_sha256": sha_file(prediction_path), "split": "valid",
             "samples": 19867, "masked_validation_samples": masked,
             "attribute_names": config["attribute_names"], "metrics": metrics,
             "passed": True, "threshold_fitting_performed": False, "test_evaluated": False}
    preserve_json(audit_path, audit)
    values = payload["logits"].astype(np.float64)
    probabilities = np.empty_like(values)
    positive = values >= 0
    probabilities[positive] = 1 / (1 + np.exp(-values[positive]))
    exponentials = np.exp(values[~positive])
    probabilities[~positive] = exponentials / (1 + exponentials)
    grid_integer = list(range(5, 96))
    grid = [value / 100 for value in grid_integer]
    thresholds, tuned_f1, baseline_f1, aps = [], [], [], []
    targets = payload["targets"]
    for column in range(24):
        labels, scores = targets[:, column], probabilities[:, column]
        assert np.unique(labels).size == 2
        baseline_f1.append(float(f1_score(labels, scores >= 0.5, zero_division=0)))
        aps.append(float(average_precision_score(labels, scores)))
        candidate_f1 = [float(f1_score(labels, scores >= threshold, zero_division=0)) for threshold in grid]
        index = min(range(len(grid)), key=lambda i: (-candidate_f1[i], abs(grid_integer[i] - 50), grid_integer[i]))
        thresholds.append(grid[index])
        tuned_f1.append(candidate_f1[index])
    assert abs(float(np.mean(baseline_f1)) - expected_f1) <= 1e-10
    assert abs(float(np.mean(aps)) - expected_map) <= 1e-10
    assert all(tuned >= baseline for tuned, baseline in zip(tuned_f1, baseline_f1))
    report = {"run_name": config["run_name"], "checkpoint": checkpoint_path.name,
              "selected_epoch": epoch, **hashes, "prediction_path": str(prediction_path),
              "prediction_sha256": audit["prediction_sha256"], "fit_split": "valid",
              "samples": 19867, "attribute_names": config["attribute_names"],
              "thresholds": thresholds, "objective": "per_attribute_F1", "grid": grid,
              "tie_break": "closest_to_0.5_then_lower", "comparison": ">=",
              "baseline_macro_f1": expected_f1, "tuned_macro_f1": float(np.mean(tuned_f1)),
              "map": expected_map, "per_attribute_f1_at_0.5": baseline_f1,
              "per_attribute_f1_tuned": tuned_f1, "per_attribute_ap": aps,
              "source_sha256": config["source_sha256"], "environment": config["environment"],
              "test_evaluated": False,
              "note": "Thresholds fitted on clean validation only. Tuned validation F1 is an in-sample fitting result, not independent evaluation. Reuse these thresholds unchanged for all clean and corrupted test conditions."}
    if kind in ("m3", "oracle"):
        report["masked_validation_samples"] = masked
        report["geometry_pipeline"] = "Cached MTCNN clean-image landmarks; shared preprocessing; retain all samples and mask invalid geometry after MLP." if kind == "m3" else config["geometry_source"]
    output = PROJECT / "runs/benchmark_preparation" / RUN_TAG / f"{config['run_name']}.json"
    preserve_json(output, report)
    print(f"{config['run_name']}: F1@0.5={expected_f1:.6f}; F1(clean-validation tuned)={report['tuned_macro_f1']:.6f}; mAP={expected_map:.6f}")
    print("Selected validation export and clean-validation threshold freeze: PASS")
    return report

def run_model(model_name):
    global evaluate_attributes
    context = prepare_run(model_name)
    evaluate_attributes = context["metric_function"]
    try:
        if model_name in ("r18", "r50"):
            train_rgb(context, PROJECT)
        else:
            trainer = train_m3 if model_name == "m3" else train_oracle
            trainer(model=context["model"], optimizer=context["optimizer"],
                    criterion=context["criterion"], config=context["config"],
                    train_dataset=context["train_dataset"], valid_dataset=context["valid_dataset"],
                    run_root=context["run_root"], device=context["device"])
        return export_and_freeze(context)
    finally:
        del context
        gc.collect()
        torch.cuda.empty_cache()

## Train, export and freeze: R18

In [ ]:
R18_REPORT = run_model("r18")

## Train, export and freeze: R50

In [ ]:
R50_REPORT = run_model("r50")

## Train, export and freeze: M3

In [ ]:
M3_REPORT = run_model("m3")

## Train, export and freeze: ORACLE

In [ ]:
ORACLE_REPORT = run_model("oracle")

## Final summary and persistent training archive

In [ ]:
reports = [R18_REPORT, R50_REPORT, M3_REPORT, ORACLE_REPORT]
print("Run                Epoch    F1@0.5    F1(clean-val tuned)    mAP")
for report in reports:
    print(f"{report['run_name']:<19}{report['selected_epoch']:>3}    {report['baseline_macro_f1']:.6f}    {report['tuned_macro_f1']:.6f}              {report['map']:.6f}")
payloads = {}
for report in reports:
    run_name = report["run_name"]
    root = PROJECT / "runs" / run_name
    for filename in ("config.json", "history.json", "best.json", "data_preparation.json", "validation_export_audit.json"):
        payloads[f"runs/{run_name}/{filename}"] = (root / filename).read_bytes()
    for relative, expected in report["source_sha256"].items():
        path = root / "source" / relative
        assert sha_file(path) == expected
        payloads[f"runs/{run_name}/source/{relative}"] = path.read_bytes()
    threshold = Path("runs/benchmark_preparation") / RUN_TAG / f"{run_name}.json"
    payloads[threshold.as_posix()] = (PROJECT / threshold).read_bytes()
payloads["summary.json"] = json_bytes(reports)
manifest = {"seed": SEED, "files": {name: hashlib.sha256(data).hexdigest() for name, data in sorted(payloads.items())},
            "checkpoints_included": False, "prediction_npz_included": False,
            "images_included": False, "test_evaluated": False}
payloads["archive_manifest.json"] = json_bytes(manifest)
archive_path = Path(f"/content/{RUN_TAG}_training_results_archive.zip")
temporary = archive_path.with_name(archive_path.name + ".partial")
with zipfile.ZipFile(temporary, "w") as archive:
    for name, data in sorted(payloads.items()):
        info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        archive.writestr(info, data)
with zipfile.ZipFile(temporary) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(payloads)
    for name, data in payloads.items():
        assert archive.read(name) == data
temporary.replace(archive_path)
archive_hash = sha_file(archive_path)
destination = PROJECT / "runs/training_archives" / archive_path.name
copy_verified(archive_path, destination, archive_hash)
print("Four-model training, validation export and threshold freeze: COMPLETE")
print("Archive saved:", destination)
print("Archive SHA256:", archive_hash)
print("Prediction NPZ and checkpoints remain in their original run directories")
print("No test evaluation performed")

## Completion and recovery

The final message confirms training and clean-validation preparation only. Formal robustness testing is a separate step using the frozen thresholds.

If the runtime disconnects, reconnect to the same recorded GPU/software environment and select Run all again. An incomplete epoch restarts from the latest completed checkpoint. Completed models verify saved records and reuse their validation predictions. Do not run duplicate training sessions for the same seed simultaneously.

Seed1 used CUDA builds different from the historical Seed0 RGB runs; reference and current environments remain recorded. This notebook requires the recorded Seed1 environment for Seed2 rather than silently changing dependencies.

Some historical Colab runs emitted DataLoader destructor warnings. This notebook preserves the loader settings and does not claim to eliminate that runtime issue. Only completed, saved checkpoints count as finished epochs.
